In [1]:
import numpy as np
import matplotlib.pyplot as plt

from troma import data_structure as ds
from troma import ConstraintSketch
from troma import bind_optimizer, get_optimizer
from troma import matching_pursuit

In [2]:
#example 2
number_spins = 6

#abstract
spectrum_bin = [
    [0,0,0,0,0,0],
    [0,1,0,1,0,1],
    [0,0,1,0,0,1],
    [1,1,1,1,1,1]
]
spectrum_pos = [ds.dit_string_to_integer(s) for s in spectrum_bin]
spectrum_val = [0.5,5,3,2]

#explicit
full_spectrum = np.zeros((2**number_spins,))
full_spectrum[spectrum_pos] = spectrum_val

#Define marginals and sketch
interaction_size = 2
constraints = ConstraintSketch.build_nearest_neighbors_sketch(number_spins, interaction_size, 2)
y = ConstraintSketch.compute_marginal((spectrum_bin, spectrum_val), constraints)

In [3]:
#Quantum methods

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2, Session

backend = AerSimulator()
with Session(backend=backend) as session:

    sampler = SamplerV2(mode=session)

    opti = bind_optimizer("qaoa", sampler=sampler,number_shots=4096)
    print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[21.    5.6 ]
 [ 9.    3.08]]


In [4]:
# Estimate hardware cost before submitting jobs

from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService
from troma.optimization._quantum_map import compute_hamiltonian, create_qaoa_circ

service = QiskitRuntimeService()
backend_name = "ibm_fez"
backend = service.backend(backend_name)

number_layers = 4
number_shots = 4096
matching_pursuit_iterations = 1

# Estimated from one AerSimulator run: mean QAOA nfev over the 2 matching-pursuit iterations was 84.5.
expected_objective_evaluations = 100 #85

# Empirical calibration from a 4096-shot Bell-state job that took about 2 seconds of QPU time.
bell_job_qpu_seconds = 2.0
bell_job_shots = 4096
empirical_seconds_per_shot = bell_job_qpu_seconds / bell_job_shots

ham_data = compute_hamiltonian(constraints, y, bit_string_length=number_spins)
qaoa_circuit = create_qaoa_circ(
    ham_data,
    num_qubits=number_spins,
    num_layers=number_layers,
 )
transpiled_qaoa_circuit = transpile(
    qaoa_circuit,
    backend=backend,
    optimization_level=1,
 )

schedule_seconds_per_shot = transpiled_qaoa_circuit.estimate_duration(target=backend.target)
circuits_per_qaoa_run = expected_objective_evaluations + 1

schedule_qaoa_run_seconds = schedule_seconds_per_shot * number_shots * circuits_per_qaoa_run
schedule_total_seconds = schedule_qaoa_run_seconds * matching_pursuit_iterations

empirical_qaoa_run_seconds = empirical_seconds_per_shot * number_shots * circuits_per_qaoa_run
empirical_total_seconds = empirical_qaoa_run_seconds * matching_pursuit_iterations

print(f"Backend: {backend.name}")
print(f"Estimated circuits per QAOA run: {circuits_per_qaoa_run}")
print()
print("Lower-bound estimate from transpiled circuit duration:")
print(f"  Duration per shot: {schedule_seconds_per_shot:.6e} s")
print(f"  Quantum time per QAOA run: {schedule_qaoa_run_seconds:.3f} s")
print(f"  Quantum time for full matching pursuit: {schedule_total_seconds:.3f} s")
print()
print("Empirical estimate calibrated from Bell-state hardware timing:")
print(f"  Effective duration per shot: {empirical_seconds_per_shot:.6e} s")
print(f"  Quantum time per QAOA run: {empirical_qaoa_run_seconds:.3f} s")
print(f"  Quantum time for full matching pursuit: {empirical_total_seconds:.3f} s")
print()
print("The empirical estimate is usually closer to the QPU time reported by IBM Runtime.")

qiskit_runtime_service.__init__:WARNING:2026-04-20 18:06:23,137: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: S0135 Quaranea. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-20 18:06:23,138: Using instance: S0135 Quaranea, plan: premium


Backend: ibm_fez
Estimated circuits per QAOA run: 101

Lower-bound estimate from transpiled circuit duration:
  Duration per shot: 4.040000e-06 s
  Quantum time per QAOA run: 1.671 s
  Quantum time for full matching pursuit: 1.671 s

Empirical estimate calibrated from Bell-state hardware timing:
  Effective duration per shot: 4.882812e-04 s
  Quantum time per QAOA run: 202.000 s
  Quantum time for full matching pursuit: 202.000 s

The empirical estimate is usually closer to the QPU time reported by IBM Runtime.


In [ ]:
#Quantum methods

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2, Session

service = QiskitRuntimeService()
backend = service.backend("ibm_marrakesh")

# with Session(backend=backend, max_time="5m") as session:

sampler = SamplerV2(mode=backend)
sampler.options.max_execution_time = 3

opti = bind_optimizer("qaoa", sampler=sampler,number_shots=512, method="COBYLA", optimizer_options={"maxiter": 30, "maxfev": 40})
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=1, step=None, interaction_size=interaction_size, optimizer=opti))

qiskit_runtime_service.__init__:WARNING:2026-04-20 18:21:34,383: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: S0135 Quaranea. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-20 18:21:34,384: Using instance: S0135 Quaranea, plan: premium


IBMRuntimeError: 'Failed to run program: \'400 Client Error: Bad Request for url: https://quantum.cloud.ibm.com/api/v1/jobs. {"errors":[{"code":1217,"message":"Session has been closed.","solution":"Increase the session `max_time` if possible, or keep session active by reducing the time between jobs.  For details, see the [Session length](https://quantum.cloud.ibm.com/docs/guides/run-jobs-session#session-length) guide.","more_info":"https://cloud.ibm.com/apidocs/quantum-computing#error-handling"}],"trace":"c665e067-7921-495e-ab33-ef3b3d41c589"}\\n\''